In [ ]:
import time
import tracemalloc
from scipy.optimize import curve_fit
import numpy as np

import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['savefig.edgecolor'] = 'white'

plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

from pathlib import Path
import importlib.util
import sys
import tempfile
import urllib.request

_reference_util_path = next(
    (
        candidate
        for base in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            base / "capitulo4" / "referencias" / "util.py",
            base / "referencias" / "util.py",
        )
        if candidate.exists()
    ),
    None,
)
if _reference_util_path is None:
    _reference_util_path = Path(tempfile.gettempdir()) / "capitulo4_reference_util.py"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/"
        "3_notas-a-mano-sobre-analisis-de-complejidad-computacional/"
        "main/capitulo4/referencias/util.py",
        _reference_util_path,
    )
_reference_spec = importlib.util.spec_from_file_location(
    "capitulo4_reference_util", _reference_util_path
)
_reference_util = importlib.util.module_from_spec(_reference_spec)
sys.modules[_reference_spec.name] = _reference_util
_reference_spec.loader.exec_module(_reference_util)
graficar_complejidad = _reference_util.graficar_complejidad
modelo_constante = _reference_util.modelo_constante
modelo_lineal = _reference_util.modelo_lineal
modelo_cuadratico = _reference_util.modelo_cuadratico


In [ ]:
def medir_tiempo(func, n_iter):
    tiempos = np.zeros(n_iter)
    for i in range(n_iter):
        a = np.random.randint(1, 1000)
        b = np.random.randint(1, 1000)
        inicio = time.perf_counter()
        func(a,b)
        tiempos[i] = time.perf_counter() - inicio
    return tiempos

def medir_memoria(func, n_iter):
    espacios = np.zeros(n_iter, dtype=int)
    for i in range(n_iter):
        a = np.random.randint(1, 1000)
        b = np.random.randint(1, 1000)
        tracemalloc.start()
        func(a,b)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        espacios[i] = peak
    return espacios

def sumar(a, b):
    return a+b



In [ ]:
n_ejecuciones = 1000
x = np.arange(n_ejecuciones)


In [ ]:

tiempos = medir_tiempo(sumar, n_ejecuciones)
params_tiempo = curve_fit(modelo_constante, x, tiempos)[0]
tiempos_ajustados = modelo_constante(x, *params_tiempo)

graficar_complejidad(
    x=x,
    y_experimental=tiempos,
    y_teorico=tiempos_ajustados,
    nombre_archivo='ejemplo_suma_dos_numeros_tiempo.png',
    ylabel="Tiempo de ejecución [s]",
    funcion="T(n)",
    legend_loc="upper right",
    y_headroom=0.40
)

In [ ]:

memorias = medir_memoria(sumar, n_ejecuciones)
params_memoria = curve_fit(modelo_constante, x, memorias)[0]
memorias_ajustadas = modelo_constante(x, *params_memoria)

graficar_complejidad(
    x=x,
    y_experimental=memorias,
    y_teorico=memorias_ajustadas,
    nombre_archivo='ejemplo_suma_dos_numeros_espacio.png',
    ylabel="Consumo de memoria [bytes]",
    funcion="S(n)",
    legend_loc="upper right",
    y_headroom=0.40
)